[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20I%20-%20The%20Port%20%26%20Container%20Terminal%20%28Problems%201-46%29/E.%20Strategic%20Design%20%26%20Long-Term%20Investment%20%28Shaping%20the%20Future%29/39.%20The%20Channel%20Dredging%20%26%20Deepening%20Investment%20Problem/P39-Tier-2_executed.ipynb) [![Open in SageMaker](https://img.shields.io/badge/Open%20in-SageMaker-orange?logo=amazon-aws)](https://aws.amazon.com/sagemaker/) [![Open in Azure](https://img.shields.io/badge/Open%20in-Azure%20Notebooks-blue?logo=microsoft-azure)](https://notebooks.azure.com/)

---


# 39. The Channel Dredging and Deepening Investment Problem

## Tier 2: Dynamic Programming Implementation

### Goal and scope

This notebook formulates a finite-horizon channel dredging problem as a dynamic program (DP). At each year, the decision is either to do nothing or to dredge one channel segment by one of a small set of possible depth increases. Dredging uses an available budget, increases channel depth, and is followed by natural sedimentation. The objective is to maximize discounted dredging benefits while satisfying the budget constraint.

The inputs are the planning horizon, channel and economic parameters, initial segment depths, and discretization settings. The outputs are the optimal value, the decisions on the optimal path from the initial state, state and computation statistics, and plots of the resulting depth and budget trajectories.

### Model assumptions

- All channel segments begin at the specified initial depth in the main example.
- At most one segment can be dredged in a year. Each year also includes a do-nothing option.
- Dredging cost is a feasibility constraint on available funds. The objective maximizes the stated dredging benefit; it does not subtract cost from benefit. Thus, the objective is not a net-profit measure.
- The benefit formula assigns a one-time benefit to each dredging action. There is no additional terminal reward for the final channel depth.
- Sedimentation is applied after the year's dredging decision. Depth is not allowed to fall below the initial depth in this model.
- Depth and budget are represented on discrete grids. This makes the state space finite, but the chosen discretization approximates the underlying continuous quantities.
- The budget carries forward between years. Each year adds a new annual allocation, and unused funds remain available, up to the maximum funds that can be allocated over the remaining horizon.

### Example data

The example has three channel segments and a five-year planning horizon. Each segment starts at 11.6 m, and the target depth is 14.0 m. Sedimentation is 0.1 m per year; dredging costs $5 million per meter per segment; the annual budget allocation is $15 million; and the annual discount rate is 5%. The depth and budget grids are specified in the parameter cell below. All quantities and units are retained as model inputs rather than treated as observed operational data.

### How the method works

A **state** records the information needed to make future decisions: the depth of every segment, the available budget, and the current period. A **decision** is an action available in that state. The recursive value function evaluates each feasible decision by adding its current benefit to the discounted value of the resulting next state. Memoization stores each computed state value so that a state reached by multiple decision paths is not solved repeatedly.

The recursion searches forward through possible decisions and terminates at the end of the planning horizon. It is equivalent to backward induction over the finite state graph: each state value is determined from values at later periods. The policy cache records the maximizing action for each nonterminal state; following those actions from the initial state extracts the optimal decision sequence for the example.

### Mathematical formulation

Let $t \in \{0,\ldots,T\}$ denote the period, let $d=(d_1,\ldots,d_n)$ be the segment-depth vector, and let $b$ be the available budget at the start of a decision period. A state is $s=(d,b,t)$. Let $A(s)$ be the decisions available in state $s$, including the do-nothing action. For a dredging decision $a$, let $c(a)$ be its cost and $r(a)$ its benefit; both are zero for the do-nothing action. Let $F(s,a)$ be the next state after applying the decision, sedimentation, budget replenishment, and discretization.

The value function is

$$V(d,b,t)=\begin{cases}
0, & t=T,\\
\displaystyle\max_{a\in A(d,b,t)}\left[r(a)+\frac{V(F(d,b,t,a))}{1+\rho}\right], & 0\le t<T.
\end{cases}$$

Here, $T$ is the number of years, $\rho$ is the annual discount rate, and $V$ is measured in the same benefit units as $r$. The feasible action set includes only dredging actions whose costs do not exceed the current budget. The discount factor is applied once for each transition to the next year, so a benefit received in year $t$ is discounted by the appropriate number of periods when evaluated from the initial state. The terminal value is zero because this model assigns benefits to dredging actions and includes no separate terminal-depth reward.

For each segment, the model adds the selected dredging increase before applying sedimentation. The resulting depth is then mapped to the depth grid. Available budget is updated by subtracting the action cost, adding the next annual allocation, applying the remaining-horizon cap, and mapping to the budget grid. The code implements these operations in `get_possible_decisions`, `apply_decision`, and `dp_recursive`.

If there are $K$ depth levels per segment, $B$ budget levels, $T$ periods, and at most $A$ actions per state, a simple upper bound on the number of state-action evaluations is proportional to $T K^n B A$. The actual number of reached states can be smaller because only states reachable from the initial state are evaluated. Increasing the number of segments or grid levels can therefore increase computation rapidly.

### Reading the results

The reported optimal value is the maximum discounted benefit under this model, not benefit minus cost. The optimal sequence is the policy path from the specified initial state; it is not a listing of every possible policy decision. State counts and cache hits describe the memoized search. Sensitivity analysis reruns the same DP with different discount rates or annual budget allocations, while keeping the other example inputs unchanged.

In [ ]:
from dataclasses import dataclass
from typing import Any, Dict, List, Optional, Tuple
import math
import time

import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

plt.style.use("seaborn-v0_8")
sns.set_palette("husl")

In [ ]:
@dataclass(frozen=True)
class ChannelState:
    """A discrete state containing depths, available funds, and period."""

    segment_depths: Tuple[float, ...]
    remaining_budget: float
    current_period: int


@dataclass(frozen=True)
class DredgingDecision:
    """One action, including the do-nothing action when segment_id is -1."""

    segment_id: int
    depth_increase: float
    cost: float
    benefit: float


@dataclass
class DPResult:
    """Value, policy, and computation statistics returned by the solver."""

    optimal_value: float
    optimal_policy: Dict[ChannelState, DredgingDecision]
    state_values: Dict[ChannelState, float]
    computation_time: float
    states_explored: int
    cache_hits: int
    optimal_sequence: List[DredgingDecision]


# Main example parameters; monetary values are in millions of US dollars.
n_segments = 3
n_years = 5
initial_depth = 11.6
target_depth = 14.0
depth_deficit = target_depth - initial_depth

annual_budget = 15.0
dredging_cost_per_meter = 5.0
discount_rate = 0.05
benefit_per_meter = 2.0

sedimentation_rate = 0.1
max_dredging_per_year = 2.0

depth_steps = 5
budget_steps = 10

print("Dynamic Programming Problem Configuration")
print(f"Segments: {n_segments}")
print(f"Planning horizon: {n_years} years")
print(f"Initial depth: {initial_depth:.1f} m; target depth: {target_depth:.1f} m")
print(f"Annual budget allocation: ${annual_budget:.1f} million")
print(f"Dredging cost: ${dredging_cost_per_meter:.1f} million per meter per segment")
print(f"Discount rate: {discount_rate:.1%} per year")
print(f"Depth grid intervals: {depth_steps}")
print(f"Budget grid intervals per annual allocation: {budget_steps}")

In [ ]:
class ChannelDredgingDP:
    """Finite-horizon recursive DP solver with memoized state values."""

    def __init__(
        self,
        n_segments: int,
        n_years: int,
        annual_budget: float,
        dredging_cost_per_meter: float,
        discount_rate: float,
        initial_depth: float,
        target_depth: float,
        benefit_per_meter: float,
        sedimentation_rate: float,
        max_dredging_per_year: float,
        depth_steps: int,
        budget_steps: int,
    ) -> None:
        if n_segments <= 0 or n_years <= 0:
            raise ValueError("The number of segments and years must be positive.")
        if depth_steps <= 0 or budget_steps <= 0:
            raise ValueError("Both discretization counts must be positive.")
        if annual_budget <= 0 or dredging_cost_per_meter <= 0:
            raise ValueError("Budget and dredging cost must be positive.")
        if discount_rate < 0 or target_depth < initial_depth:
            raise ValueError("The discount rate must be nonnegative and target depth must not be lower than initial depth.")

        self.n_segments = n_segments
        self.n_years = n_years
        self.annual_budget = annual_budget
        self.dredging_cost_per_meter = dredging_cost_per_meter
        self.discount_rate = discount_rate
        self.initial_depth = initial_depth
        self.target_depth = target_depth
        self.benefit_per_meter = benefit_per_meter
        self.sedimentation_rate = sedimentation_rate
        self.max_dredging_per_year = max_dredging_per_year
        self.depth_steps = depth_steps
        self.budget_steps = budget_steps

        self.depth_step_size = (target_depth - initial_depth) / depth_steps
        self.budget_step_size = annual_budget / budget_steps
        self.depth_deficit = target_depth - initial_depth

        self.value_cache: Dict[ChannelState, float] = {}
        self.policy_cache: Dict[ChannelState, DredgingDecision] = {}
        self.states_explored = 0
        self.cache_hits = 0

    def discretize_depth(self, depth: float) -> float:
        """Map depth to the nearest allowed level, bounded by model limits."""
        if self.depth_step_size == 0:
            return self.initial_depth
        steps = round((depth - self.initial_depth) / self.depth_step_size)
        steps = max(0, min(self.depth_steps, steps))
        return self.initial_depth + steps * self.depth_step_size

    def discretize_budget(self, budget: float) -> float:
        """Round available funds down so discretization cannot create funds."""
        steps = math.floor((budget + 1e-12) / self.budget_step_size)
        maximum_steps = self.n_years * self.budget_steps
        steps = max(0, min(maximum_steps, steps))
        return steps * self.budget_step_size

    def get_possible_decisions(
        self, state: ChannelState
    ) -> List[DredgingDecision]:
        """Enumerate feasible actions for the current state."""
        decisions = [DredgingDecision(-1, 0.0, 0.0, 0.0)]

        for segment_id, current_depth in enumerate(state.segment_depths):
            max_increase = min(
                self.max_dredging_per_year,
                self.target_depth - current_depth,
            )
            if max_increase <= 0:
                continue

            # Three candidate increases retain a small, explicit action set.
            for increase in np.linspace(0.1, max_increase, 3):
                cost = float(increase * self.dredging_cost_per_meter)
                if cost > state.remaining_budget + 1e-12:
                    continue

                remaining_deficit = self.target_depth - current_depth - increase
                benefit = increase * self.benefit_per_meter * (
                    1.0 + remaining_deficit / self.depth_deficit
                )
                decisions.append(
                    DredgingDecision(
                        segment_id=segment_id,
                        depth_increase=float(increase),
                        cost=cost,
                        benefit=float(benefit),
                    )
                )

        return decisions

    def apply_decision(
        self, state: ChannelState, decision: DredgingDecision
    ) -> ChannelState:
        """Apply dredging, sedimentation, budget replenishment, and grids."""
        new_depths = list(state.segment_depths)
        if decision.segment_id >= 0:
            new_depths[decision.segment_id] = min(
                self.target_depth,
                new_depths[decision.segment_id] + decision.depth_increase,
            )

        for index in range(self.n_segments):
            new_depths[index] = max(
                self.initial_depth,
                new_depths[index] - self.sedimentation_rate,
            )

        next_period = state.current_period + 1
        funds_after_replenishment = (
            state.remaining_budget - decision.cost + self.annual_budget
        )
        remaining_horizon_cap = self.annual_budget * (
            self.n_years - next_period
        )
        next_budget = min(funds_after_replenishment, remaining_horizon_cap)

        return ChannelState(
            segment_depths=tuple(self.discretize_depth(depth) for depth in new_depths),
            remaining_budget=self.discretize_budget(next_budget),
            current_period=next_period,
        )

    def dp_recursive(self, state: ChannelState) -> float:
        """Return the maximum discounted benefit from a state onward."""
        if state in self.value_cache:
            self.cache_hits += 1
            return self.value_cache[state]

        self.states_explored += 1
        if state.current_period >= self.n_years:
            self.value_cache[state] = 0.0
            return 0.0

        best_value = float("-inf")
        best_decision: Optional[DredgingDecision] = None

        for decision in self.get_possible_decisions(state):
            next_state = self.apply_decision(state, decision)
            value = decision.benefit + self.dp_recursive(next_state) / (
                1.0 + self.discount_rate
            )
            if value > best_value:
                best_value = value
                best_decision = decision

        if best_decision is None:
            raise RuntimeError("Every nonterminal state must have an available action.")

        self.value_cache[state] = best_value
        self.policy_cache[state] = best_decision
        return best_value

    def solve(self, initial_depths: Tuple[float, ...]) -> DPResult:
        """Solve from the specified initial depths and extract its policy path."""
        if len(initial_depths) != self.n_segments:
            raise ValueError("The initial-depth tuple must contain one value per segment.")

        self.value_cache.clear()
        self.policy_cache.clear()
        self.states_explored = 0
        self.cache_hits = 0

        initial_state = ChannelState(
            segment_depths=tuple(
                self.discretize_depth(depth) for depth in initial_depths
            ),
            remaining_budget=self.discretize_budget(self.annual_budget),
            current_period=0,
        )

        start_time = time.perf_counter()
        optimal_value = self.dp_recursive(initial_state)
        optimal_sequence = self.extract_optimal_sequence(initial_state)
        computation_time = time.perf_counter() - start_time

        return DPResult(
            optimal_value=optimal_value,
            optimal_policy=self.policy_cache.copy(),
            state_values=self.value_cache.copy(),
            computation_time=computation_time,
            states_explored=self.states_explored,
            cache_hits=self.cache_hits,
            optimal_sequence=optimal_sequence,
        )

    def extract_optimal_sequence(
        self, initial_state: ChannelState
    ) -> List[DredgingDecision]:
        """Follow cached maximizing actions from the initial state."""
        sequence = []
        state = initial_state

        while state.current_period < self.n_years:
            decision = self.policy_cache.get(state)
            if decision is None:
                raise RuntimeError("The policy is missing a reachable state.")
            sequence.append(decision)
            state = self.apply_decision(state, decision)

        return sequence


print("ChannelDredgingDP class defined.")

In [ ]:
dp_solver = ChannelDredgingDP(
    n_segments=n_segments,
    n_years=n_years,
    annual_budget=annual_budget,
    dredging_cost_per_meter=dredging_cost_per_meter,
    discount_rate=discount_rate,
    initial_depth=initial_depth,
    target_depth=target_depth,
    benefit_per_meter=benefit_per_meter,
    sedimentation_rate=sedimentation_rate,
    max_dredging_per_year=max_dredging_per_year,
    depth_steps=depth_steps,
    budget_steps=budget_steps,
)

initial_depths = tuple(initial_depth for _ in range(n_segments))
result = dp_solver.solve(initial_depths)

print("CHANNEL DREDGING DYNAMIC PROGRAMMING RESULTS")
print(f"Initial segment depths: {initial_depths} m")
print(f"Target depth: {target_depth:.1f} m")
print(f"Depth deficit per segment: {depth_deficit:.1f} m")
print(f"Optimal discounted benefit: ${result.optimal_value:.3f} million")
print(f"Computation time: {result.computation_time:.4f} seconds")
print(f"States explored: {result.states_explored:,}")
print(f"Memoized-state cache hits: {result.cache_hits:,}")
print(f"Decisions in the optimal path: {len(result.optimal_sequence)}")

print("\nOptimal decision sequence:")
for year, decision in enumerate(result.optimal_sequence, start=1):
    if decision.segment_id >= 0:
        print(
            f"Year {year}: dredge Segment {decision.segment_id + 1} by "
            f"{decision.depth_increase:.2f} m "
            f"(cost: ${decision.cost:.2f} million; "
            f"benefit: ${decision.benefit:.2f} million)"
        )
    else:
        print(f"Year {year}: no dredging")

In [ ]:
def simulate_policy(
    dp_solver: ChannelDredgingDP,
    initial_depths: Tuple[float, ...],
    optimal_sequence: List[DredgingDecision],
) -> Dict[str, Any]:
    """Replay the model transitions along the extracted optimal path."""
    state = ChannelState(
        segment_depths=tuple(
            dp_solver.discretize_depth(depth) for depth in initial_depths
        ),
        remaining_budget=dp_solver.discretize_budget(dp_solver.annual_budget),
        current_period=0,
    )

    depth_evolution = [list(state.segment_depths)]
    budget_evolution = [state.remaining_budget]
    cumulative_cost = 0.0
    cumulative_benefit = 0.0

    for decision in optimal_sequence:
        if decision.cost > state.remaining_budget + 1e-12:
            raise AssertionError("A selected decision exceeds the available budget.")
        cumulative_cost += decision.cost
        cumulative_benefit += decision.benefit
        state = dp_solver.apply_decision(state, decision)
        depth_evolution.append(list(state.segment_depths))
        budget_evolution.append(state.remaining_budget)

    return {
        "depth_evolution": depth_evolution,
        "budget_evolution": budget_evolution,
        "cumulative_cost": cumulative_cost,
        "cumulative_benefit": cumulative_benefit,
        "final_depths": list(state.segment_depths),
        "final_budget": state.remaining_budget,
    }


simulation = simulate_policy(dp_solver, initial_depths, result.optimal_sequence)
assert len(result.optimal_sequence) == n_years
assert len(simulation["depth_evolution"]) == n_years + 1
assert math.isclose(
    result.optimal_value,
    sum(
        decision.benefit / ((1.0 + discount_rate) ** year)
        for year, decision in enumerate(result.optimal_sequence)
    ),
    rel_tol=1e-9,
    abs_tol=1e-9,
)

print("OPTIMAL POLICY SIMULATION")
print(f"Total dredging cost: ${simulation['cumulative_cost']:.2f} million")
print(f"Undiscounted benefit: ${simulation['cumulative_benefit']:.2f} million")
print(f"Optimal discounted benefit: ${result.optimal_value:.3f} million")
print(
    "Budget utilization: "
    f"{simulation['cumulative_cost'] / (annual_budget * n_years):.1%} "
    "of total allocated funds"
)
print(f"Available budget at the end of the horizon: ${simulation['final_budget']:.2f} million")

print("\nFinal channel depths:")
for index, final_depth in enumerate(simulation["final_depths"], start=1):
    deficit = max(0.0, target_depth - final_depth)
    print(f"Segment {index}: {final_depth:.2f} m (target deficit: {deficit:.2f} m)")

total_improvement = sum(simulation["final_depths"]) - sum(initial_depths)
maximum_total_improvement = depth_deficit * n_segments
print(f"\nTotal depth improvement: {total_improvement:.2f} m")
print(
    "Share of the initial total depth deficit recovered: "
    f"{total_improvement / maximum_total_improvement:.1%}"
)

In [ ]:
# Check the extracted path against the action, budget, and value definitions.
state = ChannelState(
    segment_depths=tuple(
        dp_solver.discretize_depth(depth) for depth in initial_depths
    ),
    remaining_budget=dp_solver.discretize_budget(annual_budget),
    current_period=0,
)
path_value = 0.0

for decision in result.optimal_sequence:
    available_decisions = dp_solver.get_possible_decisions(state)
    assert any(
        candidate == decision for candidate in available_decisions
    ), "The selected action must be available in its state."
    assert decision.cost <= state.remaining_budget + 1e-12
    path_value += decision.benefit / (
        (1.0 + discount_rate) ** state.current_period
    )
    state = dp_solver.apply_decision(state, decision)

assert state.current_period == n_years
assert math.isclose(path_value, result.optimal_value, rel_tol=1e-9, abs_tol=1e-9)

print("Policy validation passed.")
print("Every selected action is feasible, and the path value matches the DP value.")
print(
    "This validates the extracted path and model constraints; it does not compare "
    "the DP with a different optimization method."
)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle("Channel Dredging Dynamic Programming Analysis", fontsize=16, fontweight="bold")

# Segment depth trajectories show the state changes induced by decisions and sedimentation.
ax1 = axes[0, 0]
years = list(range(n_years + 1))
depths_by_segment = list(zip(*simulation["depth_evolution"]))
for index, segment_depths in enumerate(depths_by_segment, start=1):
    ax1.plot(years, segment_depths, marker="o", linewidth=2, label=f"Segment {index}")
ax1.axhline(target_depth, color="green", linestyle="--", alpha=0.7, label="Target depth")
ax1.axhline(initial_depth, color="red", linestyle="--", alpha=0.7, label="Initial depth")
ax1.set_xlabel("Period (year)")
ax1.set_ylabel("Channel depth (m)")
ax1.set_title("Channel depth by period")
ax1.legend()
ax1.grid(True, alpha=0.3)
ax1.set_ylim(initial_depth - 0.5, target_depth + 0.5)

# The budget trajectory includes carry-over and yearly replenishment.
ax2 = axes[0, 1]
ax2.plot(years, simulation["budget_evolution"], marker="s", linewidth=2, color="blue")
ax2.set_xlabel("Period (year)")
ax2.set_ylabel("Available budget ($ million)")
ax2.set_title("Available budget by period")
ax2.grid(True, alpha=0.3)

# Costs constrain feasibility; benefits form the DP objective.
ax3 = axes[1, 0]
categories = ["Dredging cost", "Undiscounted benefit", "Discounted benefit"]
values = [
    simulation["cumulative_cost"],
    simulation["cumulative_benefit"],
    result.optimal_value,
]
bars = ax3.bar(categories, values, color=["firebrick", "seagreen", "royalblue"], alpha=0.75)
ax3.set_ylabel("Value ($ million)")
ax3.set_title("Cost constraint and benefit objective")
ax3.grid(True, axis="y", alpha=0.3)
for bar in bars:
    height = bar.get_height()
    ax3.text(
        bar.get_x() + bar.get_width() / 2,
        height,
        f"{height:.2f}",
        ha="center",
        va="bottom",
    )

# Final depths can be read against the common target and initial levels.
ax4 = axes[1, 1]
segment_names = [f"Segment {index}" for index in range(1, n_segments + 1)]
positions = np.arange(n_segments)
bars = ax4.bar(positions, simulation["final_depths"], color="steelblue", alpha=0.8, label="Final depth")
ax4.axhline(target_depth, color="green", linestyle="--", alpha=0.7, label="Target depth")
ax4.axhline(initial_depth, color="red", linestyle="--", alpha=0.7, label="Initial depth")
ax4.set_xlabel("Channel segment")
ax4.set_ylabel("Depth (m)")
ax4.set_title("Final depth by segment")
ax4.set_xticks(positions)
ax4.set_xticklabels(segment_names)
ax4.legend()
ax4.grid(True, axis="y", alpha=0.3)
for bar in bars:
    ax4.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.03,
        f"{bar.get_height():.2f}",
        ha="center",
        va="bottom",
    )

plt.tight_layout()
plt.show()

In [ ]:
# Each experiment uses the same DP recursion with a changed model parameter.
discount_rates = [0.0, 0.025, 0.05, 0.075, 0.1]
discount_results = []

for rate in discount_rates:
    solver = ChannelDredgingDP(
        n_segments=n_segments,
        n_years=n_years,
        annual_budget=annual_budget,
        dredging_cost_per_meter=dredging_cost_per_meter,
        discount_rate=rate,
        initial_depth=initial_depth,
        target_depth=target_depth,
        benefit_per_meter=benefit_per_meter,
        sedimentation_rate=sedimentation_rate,
        max_dredging_per_year=max_dredging_per_year,
        depth_steps=depth_steps,
        budget_steps=budget_steps,
    )
    sensitivity_result = solver.solve(initial_depths)
    sensitivity_simulation = simulate_policy(
        solver, initial_depths, sensitivity_result.optimal_sequence
    )
    discount_results.append(
        {
            "discount_rate": rate,
            "optimal_value": sensitivity_result.optimal_value,
            "computation_time": sensitivity_result.computation_time,
            "states_explored": sensitivity_result.states_explored,
            "total_improvement": sum(sensitivity_simulation["final_depths"])
            - sum(initial_depths),
        }
    )

print("Discount-rate sensitivity")
print(f"{'Rate':<10}{'Optimal value ($M)':<22}{'Time (s)':<12}{'States':<12}{'Depth gain (m)'}")
for item in discount_results:
    print(
        f"{item['discount_rate']:<10.1%}"
        f"{item['optimal_value']:<22.3f}"
        f"{item['computation_time']:<12.4f}"
        f"{item['states_explored']:<12,}"
        f"{item['total_improvement']:.2f}"
    )

budget_multipliers = [0.5, 0.75, 1.0, 1.25, 1.5]
budget_results = []

for multiplier in budget_multipliers:
    changed_budget = annual_budget * multiplier
    solver = ChannelDredgingDP(
        n_segments=n_segments,
        n_years=n_years,
        annual_budget=changed_budget,
        dredging_cost_per_meter=dredging_cost_per_meter,
        discount_rate=discount_rate,
        initial_depth=initial_depth,
        target_depth=target_depth,
        benefit_per_meter=benefit_per_meter,
        sedimentation_rate=sedimentation_rate,
        max_dredging_per_year=max_dredging_per_year,
        depth_steps=depth_steps,
        budget_steps=budget_steps,
    )
    sensitivity_result = solver.solve(initial_depths)
    sensitivity_simulation = simulate_policy(
        solver, initial_depths, sensitivity_result.optimal_sequence
    )
    total_allocations = changed_budget * n_years
    budget_results.append(
        {
            "multiplier": multiplier,
            "annual_budget": changed_budget,
            "optimal_value": sensitivity_result.optimal_value,
            "states_explored": sensitivity_result.states_explored,
            "total_improvement": sum(sensitivity_simulation["final_depths"])
            - sum(initial_depths),
            "budget_utilization": (
                sensitivity_simulation["cumulative_cost"] / total_allocations
                if total_allocations > 0
                else 0.0
            ),
        }
    )

print("\nAnnual-budget sensitivity")
print(f"{'Multiplier':<13}{'Annual budget ($M)':<21}{'Optimal value ($M)':<22}{'Depth gain (m)':<17}{'Utilization'}")
for item in budget_results:
    print(
        f"{item['multiplier']:<13.2f}"
        f"{item['annual_budget']:<21.2f}"
        f"{item['optimal_value']:<22.3f}"
        f"{item['total_improvement']:<17.2f}"
        f"{item['budget_utilization']:.1%}"
    )

# A state-count bound uses the depth and budget grids across the finite horizon.
depth_levels = depth_steps + 1
budget_levels = n_years * budget_steps + 1
theoretical_states = (depth_levels ** n_segments) * budget_levels * (n_years + 1)
maximum_actions_per_state = 1 + n_segments * 3
cache_hit_ratio = (
    result.cache_hits / (result.cache_hits + result.states_explored)
    if result.cache_hits + result.states_explored > 0
    else 0.0
)
average_time_per_state_ms = (
    result.computation_time / result.states_explored * 1000
    if result.states_explored > 0
    else 0.0
)

print("\nState-space and computation statistics for the main example")
print(f"Depth levels per segment: {depth_levels}")
print(f"Budget levels (upper bound): {budget_levels}")
print(f"Maximum candidate actions per state: {maximum_actions_per_state}")
print(f"Theoretical state-count upper bound: {theoretical_states:,}")
print(f"Reachable states evaluated: {result.states_explored:,}")
print(f"Memoized-state cache hits: {result.cache_hits:,}")
print(f"Cache-hit share of lookups: {cache_hit_ratio:.1%}")
print(f"Average computation time per evaluated state: {average_time_per_state_ms:.4f} ms")

## Summary and interpretation

The notebook represents a multi-period dredging problem as a finite-horizon dynamic program. A state contains the segment-depth vector, available budget, and period. At each nonterminal state, the recursion evaluates feasible actions, applies the state transition, discounts the continuation value by one period, and stores both the best value and its associated decision. Memoization avoids repeating the value calculation when the same discrete state is reached along multiple paths.

The reported policy sequence is the sequence obtained by following the cached maximizing decisions from the specified initial state. Its feasibility checks confirm that each selected action is available under the modeled budget and that the discounted benefit along the path agrees with the DP value. The plots show how the chosen actions and sedimentation affect the modeled depths, while the budget trajectory reflects carry-over and replenishment.

Sensitivity results should be read as outcomes of the same model under changed discount rates or annual budget allocations. A higher discount rate reduces the relative contribution of benefits received in later periods; it does not guarantee a particular action sequence. Additional budget can expand the feasible action set, but changes in the optimal policy depend on the state transitions, discretization, and benefit formula.

The state-count bound grows rapidly with the number of segments and grid levels. The number of states actually evaluated is the number reached by the recursive search from the initial state, while cache hits count repeated requests for states whose values are already stored. These statistics describe this finite discretized model; they are not a performance guarantee for larger instances.

The interpretation depends on the modeling assumptions. In particular, dredging cost limits feasibility but is not subtracted from the benefit objective, the benefit formula is an action-level reward, and no terminal reward is assigned to final channel depths. Depth and budget discretization also approximate continuous values. These features should be kept in mind when interpreting optimality: the policy is optimal for the specified discrete model, not automatically for every operational channel-dredging system.

Natural extensions within this modeling framework include changing the state description to represent additional operational conditions, adding constraints that reflect planning requirements, or refining the discretization when computational resources permit. Each extension changes the model and may enlarge its state space.